In [ ]:
from tqdm.auto import tqdm
from LearningAlgorithms import learnQorthogonal as lqOrth
import numpy as np
rng = np.random.default_rng(seed=40)
import torch
torch.manual_seed(212)
import json

# load the data we generated. 
# SAME DATA AS IS PASSED TO VECTOR VERSION!
data_dict = torch.load("/Users/jseid1/LearnQMethod/ScratchWork/DataGeneration/data_dict.pt")

# factors
K = 3
# Units
N = 10
T=7
# number of datasets to generate
S=100
# number of total outcomes 
m=10



#------------------- Run the LearnQ Model ----------------------------
bias_dict = {}

# calculate average bias over increasing numbers of outcomes
for m in range(2, 10):
    tqdm.write(f"m = {m}")
    # list to store the bias when we use (m) outcomes
    bias_list = []
    # m fixed within this - we iterate over data sets now
    for s in tqdm(data_dict):
        # get the targets and covariates from this data set
        treated_outcomes_list, control_outcomes_list = data_dict[s]
        # truncate to the number of outcomes we are considering in this loop (m)
        treated_outcomes_list = [outcomes[:m] for outcomes in treated_outcomes_list]
        control_outcomes_list = [outcomes[:m, :] for outcomes in control_outcomes_list]
        Q_final, w_final = lqOrth.learnQorthogonal(
            # only train on pre-treatment data
            targets = treated_outcomes_list[:T-1],      
            covariates = control_outcomes_list[:T-1],
            embedding_dim = 7,
            n_iterations = 2000,
            reg_Q = 0,
            reg_w = 0,
            verbose = False,
            init_Q = "eye"
        )

        # calculate the bias for this iteration
        true = treated_outcomes_list[T-1].squeeze()
        pred = (control_outcomes_list[T-1] @ Q_final @ w_final).squeeze()
        
        bias = torch.mean((true - pred)**2).item()
        bias_list.append(bias)
    
    # calculate mean bias
    mean_bias = sum(bias_list) / len(bias_list)
    bias_dict[m] = mean_bias

with open("dataMatrixQ.json", "w") as file:
    json.dump(bias_dict, file, indent=4)


(CVXPY) Oct 18 01:10:01 PM: Encountered unexpected exception importing solver DIFFCP:
ImportError('diffcp >= 1.0.15 is required')


FileNotFoundError: [Errno 2] No such file or directory: 'data_dict.pt'